# E7: sensitivity tests and NSGA-II benchmark

Analysis of the E7 sensitivity tests and the NSGA-II benchmark.

| Experiment | Question |
|---|---|
| E7.1 | Sensitivity to update weights (DSBPSO ρ, BPSO c1/c2 and w) |
| E7.2 | Sensitivity to swarm size N and generations G |
| E7.3 | DSBPSO vs BPSO across problem settings and sizes |
| E7.4 | Sensitivity to initial exploration and unsticking period |
| E7.5 | Sensitivity to the number of optimization scenarios |
| E7.6 | NSGA-II vs DSBPSO and BPSO across the E7.3 settings |
| E7.7 | E7.2 swarm budget grid for BPSO |
| E7.8 | E7.5 scenario counts for BPSO |

All fronts use the objectives re-evaluated on the final scenario set (`objectives_final`), so runs with different optimization scenario counts are comparable.

Each section loads its own experiment and fails with a clear message if that experiment has no completed runs yet.

In [ ]:
import sys
sys.path.insert(0, '..')

from collections import defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import wilcoxon

from src.experiments.configs import get_experiment_configs
from src.experiments.metrics import per_instance_metrics
from src.experiments.runner import load_experiment_results, load_run_result

# None = the project's results/ directory
RESULTS_DIR = None
FIG_DIR = Path('figures/e7')
FIG_DIR.mkdir(parents=True, exist_ok=True)

SIGNIFICANCE_LEVEL = 0.05

## Helpers

HVI and IGD come from `per_instance_metrics`: each seed is its own problem instance, so references are built per seed from the fronts of all configs compared on that seed, with both objectives rescaled to [0, 1] over those fronts. The HVI reference point is (-0.1, -0.1), so the largest possible HVI is 1.21. A config whose front dominates all others on a seed makes up the whole IGD reference front and gets IGD 0 there.

In [ ]:
def final_objectives(entries):
    '''Re-evaluated [E[ROV], CVaR] of a list of solutions, shape (n, 2).'''
    if not entries:
        return np.empty((0, 2))
    return np.array([e['objectives_final'] for e in entries])


def load_runs(experiment_id):
    '''Load the completed runs of an experiment as {config_name: {seed: record}}.'''
    runs = defaultdict(dict)
    for run in load_experiment_results(experiment_id, results_dir=RESULTS_DIR):
        summary = run['summary']
        if summary['status'] != 'completed':
            continue
        result = load_run_result(run['pickle_path'])
        record = {
            'front': final_objectives(result['pareto_front']),
            'archive_search': np.array([e['objectives'] for e in result['archive']]),
            'archive_final': final_objectives(result['archive']),
            'elapsed': summary['elapsed_seconds'],
        }
        if 'population_front' in result:
            record['population_front'] = final_objectives(result['population_front'])
        runs[summary['config_name']][summary['seed']] = record
    if not runs:
        raise FileNotFoundError(f'No completed runs found for {experiment_id}')
    n_runs = sum(len(by_seed) for by_seed in runs.values())
    print(f'{experiment_id}: {len(runs)} configs, {n_runs} completed runs')
    return runs


def fronts_of(runs, names, key='front'):
    '''Pick one front per run: {name: {seed: (n, 2) array}} for the names that have runs.'''
    return {
        name: {seed: record[key] for seed, record in runs[name].items()}
        for name in names if name in runs
    }


def paired_p(values_a, values_b):
    '''Two-sided Wilcoxon signed-rank p-value, paired by seed.'''
    seeds = sorted(set(values_a) & set(values_b))
    a = np.array([values_a[s] for s in seeds])
    b = np.array([values_b[s] for s in seeds])
    if len(seeds) == 0 or np.allclose(a, b):
        return 1.0
    return wilcoxon(a, b, alternative='two-sided').pvalue


def count_wins(values_a, values_b):
    '''Number of common seeds where A is strictly higher than B, and the number of common seeds.'''
    seeds = sorted(set(values_a) & set(values_b))
    return sum(values_a[s] > values_b[s] for s in seeds), len(seeds)


def print_quality_table(hvi, igd, labels, baseline=None):
    '''Mean and std of HVI and IGD per config, with paired tests against the baseline if given.'''
    labels = [label for label in labels if label in hvi]
    n_tests = max(1, len(labels) - 1)
    header = 'Config                      HVI mean   HVI std   IGD mean   IGD std'
    if baseline is not None:
        header += '   HVI wins   p(HVI)    p(IGD)'
    print(header)
    for label in labels:
        h = np.array(list(hvi[label].values()))
        g = np.array(list(igd[label].values()))
        row = f'{label:26s} {h.mean():9.3f} {h.std():9.3f} {g.mean():10.4f} {g.std():9.4f}'
        if baseline is not None and label != baseline and baseline in hvi:
            won, n = count_wins(hvi[label], hvi[baseline])
            p_hvi = paired_p(hvi[label], hvi[baseline])
            p_igd = paired_p(igd[label], igd[baseline])
            flag = '*' if p_hvi < SIGNIFICANCE_LEVEL / n_tests else ' '
            row += f'   {won:3d}/{n:<3d}  {p_hvi:8.5f}{flag} {p_igd:8.5f}'
        print(row)
    if baseline is not None:
        print(
            f'Tests are against {baseline}; * = significant on HVI at '
            f'{SIGNIFICANCE_LEVEL} with Bonferroni correction over {n_tests} tests.'
        )


def print_pairwise_tests(hvi, igd, labels):
    '''Paired tests between every pair of configs on HVI and IGD.'''
    pairs = list(combinations([label for label in labels if label in hvi], 2))
    threshold = SIGNIFICANCE_LEVEL / len(pairs)
    print(f'Pair                                       HVI: A wins   p           IGD: A wins   p')
    for a, b in pairs:
        hvi_won, n = count_wins(hvi[a], hvi[b])
        igd_won, _ = count_wins(igd[b], igd[a])
        p_hvi = paired_p(hvi[a], hvi[b])
        p_igd = paired_p(igd[a], igd[b])
        hvi_flag = '*' if p_hvi < threshold else ' '
        igd_flag = '*' if p_igd < threshold else ' '
        pair = f'{a} vs {b}'
        print(
            f'{pair:42s} {hvi_won:5d}/{n:<5d} {p_hvi:9.2e}{hvi_flag}'
            f'   {igd_won:5d}/{n:<5d} {p_igd:9.2e}{igd_flag}'
        )
    print(
        f'A = first config of the pair. * = significant at {SIGNIFICANCE_LEVEL} '
        f'with Bonferroni correction over {len(pairs)} pairs (p < {threshold:.4f}).'
    )


def boxplots(hvi, igd, labels, tick_labels, filename, colors=None):
    '''Side-by-side HVI and IGD box plots, saved as PDF.'''
    shown = [(label, tick) for label, tick in zip(labels, tick_labels) if label in hvi]
    if colors is None:
        colors = ['steelblue'] * len(shown)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    hvi_data = [np.array(list(hvi[label].values())) for label, _ in shown]
    igd_data = [np.array(list(igd[label].values())) for label, _ in shown]
    panels = [
        (axes[0], hvi_data, 'Normalised HVI (higher = better)'),
        (axes[1], igd_data, 'Normalised IGD (lower = better)'),
    ]
    for ax, data, ylabel in panels:
        boxes = ax.boxplot(data, patch_artist=True, widths=0.6)
        for patch, color in zip(boxes['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_xticks(range(1, len(shown) + 1))
        ax.set_xticklabels([tick for _, tick in shown], rotation=30, ha='right')
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIG_DIR / filename, bbox_inches='tight')
    plt.show()

## 1. E7.1 — Update weights

Three one-at-a-time sweeps on the baseline problem: the DSBPSO importance ratio ρ = i_p / i_g, the BPSO ratio c1 / c2 (with c1 + c2 = 4, w = 0.9), and the BPSO constant inertia weight w. References are built per seed over all 13 configs, since they solve the same problem on a given seed.

In [ ]:
e7_1 = load_runs('E7.1')
e7_1_names = [name for name, _ in get_experiment_configs('E7.1')]
hvi_1, igd_1 = per_instance_metrics(fronts_of(e7_1, e7_1_names))

rho_names = [name for name in e7_1_names if name.startswith('dsbpso_rho_')]
cratio_names = [name for name in e7_1_names if name.startswith('bpso_cratio_')]
w_names = [name for name in e7_1_names if name.startswith('bpso_w_')] + ['bpso_cratio_1p0']

print('DSBPSO importance ratio (baseline 2.0)')
print_quality_table(hvi_1, igd_1, rho_names, 'dsbpso_rho_2p0')
print()
print('BPSO c1 / c2 (baseline 1.0)')
print_quality_table(hvi_1, igd_1, cratio_names, 'bpso_cratio_1p0')
print()
print('BPSO constant inertia weight (baseline 0.9 = bpso_cratio_1p0)')
print_quality_table(hvi_1, igd_1, w_names, 'bpso_cratio_1p0')
print()
print('Every BPSO setting against the DSBPSO baseline')
print_quality_table(
    hvi_1, igd_1, ['dsbpso_rho_2p0'] + cratio_names + w_names[:-1], 'dsbpso_rho_2p0',
)

In [ ]:
tick_labels_1 = [name.replace('dsbpso_rho_', 'ρ=').replace('p', '.') for name in rho_names]
boxplots(hvi_1, igd_1, rho_names, tick_labels_1, 'e7_1_dsbpso_rho_boxplots.pdf')

## 2. E7.2 — Swarm size and generations

DSBPSO on the baseline problem for every combination of N and G. The baseline is N = 200, G = 2000. If the default budget is sufficient, the cells with a larger budget should show no significant gain over it.

In [ ]:
e7_2 = load_runs('E7.2')
e7_2_configs = get_experiment_configs('E7.2')
e7_2_names = [name for name, _ in e7_2_configs]
hvi_2, igd_2 = per_instance_metrics(fronts_of(e7_2, e7_2_names))

print_quality_table(hvi_2, igd_2, e7_2_names, 'n200_g2000')

print()
print('Mean wall-clock time per run (s)')
for name in e7_2_names:
    if name in e7_2:
        times = [record['elapsed'] for record in e7_2[name].values()]
        print(f'  {name:14s} {np.mean(times):8.0f}')

In [ ]:
swarm_sizes = sorted({overrides['pso.n_particles'] for _, overrides in e7_2_configs})
generations = sorted({overrides['pso.max_iterations'] for _, overrides in e7_2_configs})

hvi_grid = np.full((len(swarm_sizes), len(generations)), np.nan)
igd_grid = np.full((len(swarm_sizes), len(generations)), np.nan)
for name, overrides in e7_2_configs:
    if name not in hvi_2:
        continue
    row = swarm_sizes.index(overrides['pso.n_particles'])
    col = generations.index(overrides['pso.max_iterations'])
    hvi_grid[row, col] = np.mean(list(hvi_2[name].values()))
    igd_grid[row, col] = np.mean(list(igd_2[name].values()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
panels = [
    (axes[0], hvi_grid, 'Mean normalised HVI', '.3f'),
    (axes[1], igd_grid, 'Mean normalised IGD', '.3f'),
]
for ax, grid, title, fmt in panels:
    image = ax.imshow(grid, cmap='viridis', aspect='auto')
    for row in range(len(swarm_sizes)):
        for col in range(len(generations)):
            if not np.isnan(grid[row, col]):
                ax.text(col, row, format(grid[row, col], fmt), ha='center', va='center', color='white')
    ax.set_xticks(range(len(generations)))
    ax.set_xticklabels(generations)
    ax.set_yticks(range(len(swarm_sizes)))
    ax.set_yticklabels(swarm_sizes)
    ax.set_xlabel('Generations G')
    ax.set_ylabel('Swarm size N')
    ax.set_title(title)
    fig.colorbar(image, ax=ax)
plt.tight_layout()
fig.savefig(FIG_DIR / 'e7_2_budget_heatmaps.pdf', bbox_inches='tight')
plt.show()

## 3. E7.3 and E7.6 — Algorithms across problem settings

Every problem setting is a different problem, so HVI and IGD references are built per setting and seed, over the algorithms run on it. E7.3 gives DSBPSO and BPSO (w = 0.9). E7.6 gives NSGA-II, with two fronts per run: the final population (standard NSGA-II) and the passive external archive (same bookkeeping as the PSO archive). NSGA-II is included when E7.6 results exist.

The main comparison pools three fronts per setting and seed: DSBPSO, BPSO and the NSGA-II final population (`hvi_3`, `igd_3`). The NSGA-II archive is a separate check with its own pool, in which it replaces the final population (`hvi_3_archive`, `igd_3_archive`). Pooling both NSGA-II fronts together would put NSGA-II's solutions in the IGD reference front twice.

In [ ]:
e7_3 = load_runs('E7.3')
try:
    e7_6 = load_runs('E7.6')
except FileNotFoundError:
    e7_6 = {}
    print('E7.6: no completed runs, NSGA-II left out')

suffix = '_dsbpso'
settings = [
    name[:-len(suffix)] for name, _ in get_experiment_configs('E7.3')
    if name.endswith(suffix)
]
algorithms = ['DSBPSO', 'BPSO', 'NSGA-II']

hvi_3 = {}
igd_3 = {}
hvi_3_archive = {}
igd_3_archive = {}
for setting in settings:
    fronts = {}
    if f'{setting}_dsbpso' in e7_3:
        fronts['DSBPSO'] = fronts_of(e7_3, [f'{setting}_dsbpso'])[f'{setting}_dsbpso']
    if f'{setting}_bpso' in e7_3:
        fronts['BPSO'] = fronts_of(e7_3, [f'{setting}_bpso'])[f'{setting}_bpso']
    if 'DSBPSO' not in fronts:
        continue
    nsga2_name = f'{setting}_nsga2'
    if nsga2_name in e7_6:
        archive_fronts = dict(fronts)
        archive_fronts['NSGA-II archive'] = fronts_of(e7_6, [nsga2_name])[nsga2_name]
        hvi_3_archive[setting], igd_3_archive[setting] = per_instance_metrics(archive_fronts)
        fronts['NSGA-II'] = fronts_of(e7_6, [nsga2_name], key='population_front')[nsga2_name]
    if len(fronts) < 2:
        continue
    hvi_3[setting], igd_3[setting] = per_instance_metrics(fronts)

print(f'{len(hvi_3)} of {len(settings)} settings have DSBPSO and at least one other algorithm')

In [ ]:
def print_comparison(metric, other, higher_is_better, reference='DSBPSO'):
    '''The reference algorithm against one other algorithm, one row per problem setting.'''
    compared = [
        setting for setting in settings
        if setting in metric and other in metric[setting] and reference in metric[setting]
    ]
    if not compared:
        print(f'No settings with {other}')
        return
    threshold = SIGNIFICANCE_LEVEL / len(compared)
    n_better = 0
    n_significant = 0
    n_significant_other = 0
    print(f'Setting                {reference:>11s} mean   {other:>16s} mean   {reference:>6s} wins   p')
    for setting in compared:
        own = metric[setting][reference]
        rival = metric[setting][other]
        if higher_is_better:
            won, n = count_wins(own, rival)
        else:
            won, n = count_wins(rival, own)
        p = paired_p(own, rival)
        mean_own = np.mean(list(own.values()))
        mean_rival = np.mean(list(rival.values()))
        own_better = mean_own > mean_rival if higher_is_better else mean_own < mean_rival
        n_better += own_better
        n_significant += own_better and p < threshold
        n_significant_other += (not own_better) and p < threshold
        flag = '*' if p < threshold else ' '
        print(f'{setting:22s} {mean_own:13.4g} {mean_rival:23.4g}   {won:5d}/{n:<5d} {p:9.5f}{flag}')
    print(
        f'{reference} has the better mean on {n_better} of {len(compared)} settings, '
        f'significantly so on {n_significant}. {other} is significantly better on '
        f'{n_significant_other} (Bonferroni over {len(compared)} settings).'
    )


for other in algorithms[1:]:
    print(f'=== HVI: DSBPSO vs {other} ===')
    print_comparison(hvi_3, other, higher_is_better=True)
    print()
    print(f'=== IGD: DSBPSO vs {other} ===')
    print_comparison(igd_3, other, higher_is_better=False)
    print()

print('=== HVI: NSGA-II vs BPSO ===')
print_comparison(hvi_3, 'BPSO', higher_is_better=True, reference='NSGA-II')
print()
print('=== IGD: NSGA-II vs BPSO ===')
print_comparison(igd_3, 'BPSO', higher_is_better=False, reference='NSGA-II')
print()

print('=== HVI: DSBPSO vs NSGA-II archive (own pool) ===')
print_comparison(hvi_3_archive, 'NSGA-II archive', higher_is_better=True)
print()
print('=== IGD: DSBPSO vs NSGA-II archive (own pool) ===')
print_comparison(igd_3_archive, 'NSGA-II archive', higher_is_better=False)
print()

print('=== HVI mean [std] by problem size ===')
for setting in ['size_5f_2h', 'baseline', 'size_18f_3h', 'size_18f_6h']:
    if setting not in hvi_3:
        continue
    cells = []
    for algorithm in algorithms:
        if algorithm in hvi_3[setting]:
            values = np.array(list(hvi_3[setting][algorithm].values()))
            cells.append(f'{algorithm} {values.mean():.3f} [{values.std():.3f}]')
    print(f'{setting:14s} ' + '   '.join(cells))

In [ ]:
compared_settings = [setting for setting in settings if setting in hvi_3]
bars = [
    ('BPSO', hvi_3, 'coral'),
    ('NSGA-II', hvi_3, 'seagreen'),
    ('NSGA-II archive', hvi_3_archive, 'gray'),
]

fig, ax = plt.subplots(figsize=(14, 5))
width = 0.8 / len(bars)
for offset, (other, metric, color) in enumerate(bars):
    gains = []
    for setting in compared_settings:
        if setting not in metric or other not in metric[setting]:
            gains.append(np.nan)
            continue
        mean_dsbpso = np.mean(list(metric[setting]['DSBPSO'].values()))
        mean_rival = np.mean(list(metric[setting][other].values()))
        gains.append(100 * (mean_dsbpso - mean_rival) / mean_rival)
    positions = np.arange(len(compared_settings)) + offset * width
    ax.bar(positions, gains, width=width, color=color, label=f'vs {other}')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(np.arange(len(compared_settings)) + width)
ax.set_xticklabels(compared_settings, rotation=90)
ax.set_ylabel('DSBPSO mean HVI gain (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / 'e7_3_hvi_gain_per_setting.pdf', bbox_inches='tight')
plt.show()

## 4. E7.4 — DSBPSO exploration and unsticking period

Initial exploration i_s = factor / D (baseline factor 10) and the unsticking period as a share of G (baseline 1 % → 10 %). The baseline of both sweeps is `is_10p0`.

In [ ]:
e7_4 = load_runs('E7.4')
e7_4_names = [name for name, _ in get_experiment_configs('E7.4')]
hvi_4, igd_4 = per_instance_metrics(fronts_of(e7_4, e7_4_names))

print_quality_table(hvi_4, igd_4, e7_4_names, 'is_10p0')
boxplots(hvi_4, igd_4, e7_4_names, e7_4_names, 'e7_4_dsbpso_parameter_boxplots.pdf')

## 5. E7.5 — Optimization scenario count and estimation uncertainty

The search evaluates portfolios on S scenarios and the archive is then re-evaluated on the final scenario set, which is identical across S for a given seed. Three things are reported per S:

- **Front quality** on the final scenario set (HVI, IGD).
- **Estimation error** of the search-time objectives: the relative difference between the S-scenario estimate and the final estimate, averaged over the archive. The signed value is the bias (positive = the search was optimistic), the absolute value is the error.
- **Retained share**: the share of archive members still non-dominated after re-evaluation.

In [ ]:
def print_estimation_table(runs, names):
    '''Search-time vs final objective estimates per config, averaged over runs.'''
    print(
        'Config                ROV bias %   ROV error %   CVaR bias %   CVaR error %'
        '   retained %   time (s)'
    )
    for name in names:
        if name not in runs:
            continue
        bias = []
        error = []
        retained = []
        times = []
        for record in runs[name].values():
            if len(record['archive_final']) == 0:
                continue
            relative = (
                (record['archive_search'] - record['archive_final'])
                / np.abs(record['archive_final'])
            )
            bias.append(100 * relative.mean(axis=0))
            error.append(100 * np.abs(relative).mean(axis=0))
            retained.append(100 * len(record['front']) / len(record['archive_final']))
            times.append(record['elapsed'])
        bias = np.mean(bias, axis=0)
        error = np.mean(error, axis=0)
        print(
            f'{name:20s} {bias[0]:11.2f} {error[0]:13.2f} {bias[1]:13.2f} {error[1]:14.2f}'
            f' {np.mean(retained):12.1f} {np.mean(times):10.0f}'
        )


e7_5 = load_runs('E7.5')
e7_5_names = [name for name, _ in get_experiment_configs('E7.5')]
hvi_5, igd_5 = per_instance_metrics(fronts_of(e7_5, e7_5_names))

print_quality_table(hvi_5, igd_5, e7_5_names, 's200')
print()
print_estimation_table(e7_5, e7_5_names)

tick_labels_5 = [name.replace('s', 'S=') for name in e7_5_names]
boxplots(hvi_5, igd_5, e7_5_names, tick_labels_5, 'e7_5_scenario_count_boxplots.pdf')

### Estimation uncertainty in the E4.1 runs

The same estimation table for the E4.1 runs (S = 200), which needs no new runs.

In [ ]:
e4_1 = load_runs('E4.1')
print_estimation_table(e4_1, ['dsbpso', 'bpso_constant_w', 'bpso_increasing_w'])

## 6. E1.1, E2.1 and E4.1 with per-seed normalised references

The E1.1, E2.1 and E4.1 results recomputed with the same metric as the E7 experiments. No new runs are needed. Notebooks 07, 08 and 10 use pooled references instead.

In [ ]:
earlier_experiments = [
    (
        'E1.1: encoding', load_runs('E1.1'),
        ['all_gray', 'all_binary', 'assignment_binary', 'timing_binary', 'wells_binary'],
        ['All Gray', 'All binary', 'Assign. binary', 'Timing binary', 'Wells binary'],
        ['steelblue', 'coral', 'seagreen', 'orchid', 'goldenrod'],
        'e1_1_boxplots.pdf',
    ),
    (
        'E2.1: constraint handling', load_runs('E2.1'),
        ['capping', 'cdp'],
        ['FCFS capping', 'CDP'],
        ['steelblue', 'coral'],
        'e2_1_boxplots.pdf',
    ),
    (
        'E4.1: stickiness', e4_1,
        ['dsbpso', 'bpso_constant_w', 'bpso_increasing_w'],
        ['CMO-BPSO', 'BPSO (w=0.9)', 'BPSO (w: 0.4→0.9)'],
        ['steelblue', 'coral', 'seagreen'],
        'e4_1_boxplots.pdf',
    ),
]

for title, runs, names, tick_labels, colors, filename in earlier_experiments:
    hvi_exp, igd_exp = per_instance_metrics(fronts_of(runs, names))
    print(f'=== {title} ===')
    print_quality_table(hvi_exp, igd_exp, names)
    print()
    print_pairwise_tests(hvi_exp, igd_exp, names)
    print()
    print('Mean wall-clock time per run (s)')
    for name in names:
        times = [record['elapsed'] for record in runs[name].values()]
        print(f'  {name:20s} {np.mean(times):8.0f}')
    boxplots(hvi_exp, igd_exp, names, tick_labels, filename, colors=colors)

## 7. E7.7 and E7.8 — BPSO swarm budget and scenario count

The swarm-budget grid of E7.2 and the scenario counts of E7.5, repeated for the two BPSO variants of E4.1 (constant w = 0.9 and w: 0.4 → 0.9) on the same seeds. All configs of a sweep solve the same problem on a given seed, so references are built per seed over DSBPSO and both BPSO variants at every budget (48 configs) or every scenario count (12 configs). Values are therefore comparable across budgets and across algorithms.

Two questions per sweep: does DSBPSO beat BPSO at the same budget or scenario count, and is BPSO itself sensitive to it?

In [ ]:
BPSO_VARIANTS = ['bpso_constant_w', 'bpso_increasing_w']


def print_dsbpso_against_bpso(hvi, igd, dsbpso_names):
    '''DSBPSO against each BPSO variant at the same budget or scenario count, paired by seed.'''
    threshold = SIGNIFICANCE_LEVEL / (len(dsbpso_names) * len(BPSO_VARIANTS))
    n_significant = 0
    print('Setting        Variant              DSBPSO HVI   BPSO HVI   DSBPSO wins   p(HVI)     p(IGD)')
    for name in dsbpso_names:
        for variant in BPSO_VARIANTS:
            rival = f'{variant}_{name}'
            won, n = count_wins(hvi[name], hvi[rival])
            p_hvi = paired_p(hvi[name], hvi[rival])
            p_igd = paired_p(igd[name], igd[rival])
            mean_dsbpso = np.mean(list(hvi[name].values()))
            mean_rival = np.mean(list(hvi[rival].values()))
            significant = mean_dsbpso > mean_rival and p_hvi < threshold
            n_significant += significant
            flag = '*' if p_hvi < threshold else ' '
            print(
                f'{name:14s} {variant:18s} {mean_dsbpso:11.3f} {mean_rival:10.3f}'
                f'   {won:5d}/{n:<5d} {p_hvi:9.2e}{flag} {p_igd:9.2e}'
            )
    n_tests = len(dsbpso_names) * len(BPSO_VARIANTS)
    print(
        f'DSBPSO is significantly better on HVI in {n_significant} of {n_tests} comparisons '
        f'(Bonferroni over {n_tests} tests, p < {threshold:.5f}).'
    )


sweeps = [
    ('Swarm budget (E7.2 and E7.7)', e7_2, load_runs('E7.7'), e7_2_names, 'n200_g2000'),
    ('Scenario count (E7.5 and E7.8)', e7_5, load_runs('E7.8'), e7_5_names, 's200'),
]

for title, dsbpso_runs, bpso_runs, dsbpso_names, baseline in sweeps:
    fronts = fronts_of(dsbpso_runs, dsbpso_names)
    fronts.update(fronts_of(bpso_runs, list(bpso_runs)))
    hvi_sweep, igd_sweep = per_instance_metrics(fronts)

    print(f'=== {title}: DSBPSO against BPSO ===')
    print_dsbpso_against_bpso(hvi_sweep, igd_sweep, dsbpso_names)
    print()
    print(f'=== {title}: DSBPSO against its default ===')
    print_quality_table(hvi_sweep, igd_sweep, dsbpso_names, baseline)
    print()
    for variant in BPSO_VARIANTS:
        variant_names = [f'{variant}_{name}' for name in dsbpso_names]
        print(f'=== {title}: {variant} against its default ===')
        print_quality_table(hvi_sweep, igd_sweep, variant_names, f'{variant}_{baseline}')
        print()
        print(f'=== {title}: every {variant} setting against the DSBPSO default ===')
        print_quality_table(hvi_sweep, igd_sweep, [baseline] + variant_names, baseline)
        print()